In [1]:
import os
os.chdir("..")

In [202]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)


In [203]:
MODEL_NAME = "models/policy_assistant_v2/merged_model"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [207]:
model=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.float16
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [132]:
from datasets import load_dataset
dataset=load_dataset(
    "json",
    data_files={
        "validation":"data/instruction_validation.jsonl",
        "test":"data/instruction_test.jsonl",
    }
)

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [133]:
dataset

DatasetDict({
    validation: Dataset({
        features: ['task', 'source_id', 'messages'],
        num_rows: 78
    })
    test: Dataset({
        features: ['task', 'source_id', 'messages'],
        num_rows: 78
    })
})

In [134]:
def prepare_for_eval(example):
    message=example['messages']
    
    user=message[0]
    assistant=message[1]
    
    model_ips=tokenizer.apply_chat_template([user],add_generation_prompt=True,return_tensors="pt")
    
    labels=assistant['content']
    model_ips['labels']=labels
    return model_ips

In [136]:
eval_dataset=dataset.map(prepare_for_eval,remove_columns=dataset['test'].column_names)

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

In [137]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [205]:
model.to(device)
model.eval()
def prepare_rouge(example):
    input_ids=torch.tensor(example['input_ids'])
    attention_mask=torch.tensor(example['attention_mask'])
    
    with torch.no_grad():
        op=model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=300,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    op=op[0]
    op=op[input_ids.shape[-1]:]
    
    pred_text=tokenizer.decode(op,skip_special_tokens=True)
    return {
        'pred_text':pred_text
    }    

In [206]:
prepare_rouge(eval_dataset['test'][1])

c:\Users\PratushPc\miniconda3\envs\policy_llm\Lib\site-packages\transformers\generation\utils.py:2613: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


{'pred_text': '{"BasicAccountInfo": [], "Biometric": [], "CommunicationProv": [], "ContactInfo": ["contact information"], "ContentPreferences": [], "Demographic": [], "DeviceInfo": ["your device model", "operating system", "IP address", "system language", "device ID", "user ID", "information such as your device ID and user ID"], "Financial": [], "GeoInfo": [], "HealthFitness": [], "Images": [], "InternetHistory": [], "Metadata": [], "Payment": ["payment card information", "billing", "delivery", "contact information"], "Performance": ["crash reports", "performance logs"], "Purchase": ["items you purchased"], "Settings": [], "UsageData": [], "UserGenerated": ["the information you send us"], "UserProfileInfo": []}'}

In [140]:
rouge_dataset=eval_dataset.map(prepare_rouge,remove_columns=eval_dataset['test'].column_names)

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

c:\Users\PratushPc\miniconda3\envs\policy_llm\Lib\site-packages\transformers\generation\utils.py:2613: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(
c:\Users\PratushPc\miniconda3\envs\policy_llm\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Map:   0%|          | 0/78 [00:00<?, ? examples/s]

In [184]:
from training.evaluate_metrics import compute_metrics

validation_results=compute_metrics(
    rouge_dataset['validation']['pred_text'][:],
    eval_dataset['validation']['labels'][:]
)

test_results = compute_metrics(
    rouge_dataset["test"]["pred_text"][:], 
    eval_dataset["test"]["labels"][:]
)
validation_results = {k: round(float(v), 4) for k, v in validation_results.items()}
test_results = {k: round(float(v), 4) for k, v in test_results.items()}


In [201]:
print(f"{'-'*50} Validation ROUGE {'-'*54}\n{validation_results}")
print()
print(f"{'-'*50} Test ROUGE {'-'*60}\n{test_results}")

-------------------------------------------------- Validation ROUGE ------------------------------------------------------
{'rouge1': 0.6252, 'rouge2': 0.4777, 'rougeL': 0.545}

-------------------------------------------------- Test ROUGE ------------------------------------------------------------
{'rouge1': 0.6071, 'rouge2': 0.4718, 'rougeL': 0.527}
